<a href="https://colab.research.google.com/github/Harikeshkaant/DML/blob/main/Final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1)Import libraires

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LogisticRegression, LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm

np.random.seed(42)


# 2. SYNTHETIC DATA GENERATION

In [ ]:
def generate_data(n=2000, p=10):
    X = np.random.normal(0, 1, (n, p))

    # True CATE (heterogeneous effect)
    true_theta = 2 + X[:, 0] + 0.5 * (X[:, 1] ** 2)

    # Propensity score (non-linear)
    logits = X[:, 0] - 0.5 * X[:, 1] + 0.3 * (X[:, 2] ** 2)
    prob = 1 / (1 + np.exp(-logits))
    T = np.random.binomial(1, prob)

    # Outcome function (non-linear)
    g_X = X[:, 0]**2 + np.sin(X[:, 1]) + 0.5 * X[:, 2]

    # Outcome
    Y = true_theta * T + g_X + np.random.normal(0, 1, n)

    return X, T, Y, true_theta


# Generate data
X, T, Y, true_theta = generate_data()

# 3. DML WITH CROSS-FITTING (K=5)

In [ ]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)

Y_residuals = np.zeros(len(Y))
T_residuals = np.zeros(len(T))

for train_idx, test_idx in kf.split(X):

    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]
    T_train, T_test = T[train_idx], T[test_idx]

# Stage 1: Nuisance Models

    # Outcome model (Lasso)
    model_y = Lasso(alpha=0.01)
    model_y.fit(X_train, Y_train)
    m_hat = model_y.predict(X_test)

    # Propensity model (Logistic Regression)
    model_t = LogisticRegression(max_iter=1000)
    model_t.fit(X_train, T_train)
    e_hat = model_t.predict_proba(X_test)[:, 1]

 # Stage 2: Residuals

    Y_residuals[test_idx] = Y_test - m_hat
    T_residuals[test_idx] = T_test - e_hat



# 4. FINAL STAGE: ATE ESTIMATION

In [ ]:
X_final = sm.add_constant(T_residuals)
model_final = sm.OLS(Y_residuals, X_final).fit()

tau_hat = model_final.params[1]
std_error = model_final.bse[1]
p_value = model_final.pvalues[1]

print("===== ATE ESTIMATION =====")
print(f"Estimated ATE (tau): {tau_hat:.4f}")
print(f"Standard Error: {std_error:.4f}")
print(f"P-value: {p_value:.4f}")

===== ATE ESTIMATION =====
Estimated ATE (tau): 2.3463
Standard Error: 0.1089
P-value: 0.0000


# 5. CATE ESTIMATION (HETEROGENEOUS)

In [ ]:
# Estimate CATE using interaction model
X_cate = np.column_stack((T_residuals, X))
X_cate = sm.add_constant(X_cate)

model_cate = sm.OLS(Y_residuals, X_cate).fit()
theta_hat = model_cate.predict(X_cate)


# 6. EVALUATION

In [ ]:
bias = np.mean(theta_hat - true_theta)
rmse = np.sqrt(mean_squared_error(true_theta, theta_hat))

print("\n===== CATE EVALUATION =====")
print(f"Bias: {bias:.4f}")
print(f"RMSE: {rmse:.4f}")



===== CATE EVALUATION =====
Bias: -2.5086
RMSE: 3.0000


# 7. OPTIONAL: SAVE RESULTS


In [ ]:
results_df = pd.DataFrame({
    "True_CATE": true_theta,
    "Estimated_CATE": theta_hat
})

results_df.to_csv("cate_results.csv", index=False)

print("\nResults saved to cate_results.csv")


Results saved to cate_results.csv


# Summary of my project

This project implements and evaluates a Double Machine Learning (DML) framework for causal inference to estimate Heterogeneous Treatment Effects (HTE) using programmatically generated synthetic data. The dataset simulates a randomized controlled trial (RCT) with high-dimensional confounders, a non-linear outcome function, and a treatment assignment mechanism dependent on features.

DML combines machine learning models (e.g., Lasso, Gradient Boosting) with orthogonal moment conditions to ensure valid statistical inference, even with complex nuisance functions like propensity scores and outcome models.

The core method uses cross-fitting (K=5 folds):

Stage 1: Predict outcome (Y) and treatment (T) from features (X) using ML models
Stage 2: Compute orthogonal residuals
𝑌
~
,
𝑇
~
Y
~
,
T
~
 for debiasing

A final linear model
𝑌
~
∼
𝑇
~
Y
~
∼
T
~
 is used to estimate the Conditional Average Treatment Effect (CATE) and Average Treatment Effect (τ), along with standard errors, confidence intervals, and p-values.

The estimated CATE is compared with the true (ground truth) CATE from the synthetic data using bias and RMSE, evaluating the efficiency and accuracy of the DML estimator.